# Apache Ozone Deep-Dive: Architettura, Stress Test & API S3

Questo laboratorio analizza in modo approfondito e interattivo **Apache Ozone (v2.2.1)**, il sistema di object storage distribuito di nuova generazione progettato per superare i limiti architetturali di Apache HDFS.

---

### Obiettivi del Laboratorio
1. **Gerarchia a tre livelli**: Volume -> Bucket -> Key (disaccoppiamento rispetto al namespace monolitico HDFS)
2. **Benchmark "Small Files Problem"**: Analisi delle prestazioni di ingestione e gestione dei metadati basata su **RocksDB** nell'Ozone Manager (OM)
3. **Compatibilità Nativa con l'API S3**: Interazione con l'Ozone S3 Gateway tramite **Boto3** e mapping bidirezionale con i volumi Ozone
4. **Monitoraggio e Telemetria**: Ispezione dello stato del cluster mediante la dashboard Apache Ozone Recon

---

# PARTE 1: Volume, Bucket, Key

In Hadoop HDFS classico, il filesystem è strutturato come un albero monolitico (`/cartella/sottocartella/file`), in cui la totalità dei metadati risiede nella **RAM del NameNode**.

Apache Ozone introduce un'architettura **gerarchica a tre livelli, disaccoppiata e multi-tenant**:

```
Volume (Amministrazione, quote di storage e namespace, tenancy)
 └── Bucket (Contenitore logico, direttamente mappato all'interfaccia S3)
      └── Key (Oggetto dati atomico persistito nei blocchi distribuiti)
```

- **Volume**: Entità amministrativa creata tipicamente dagli operatori di sistema per organizzare dipartimenti, progetti o tenant (es. `/finance`, `/marketing`, `/volume-lab`). Definisce quote di storage, limiti di namespace e policy di accesso.
- **Bucket**: Entità equivalente a un bucket Amazon S3, contenuta all'interno di un volume. Gli utenti possono creare e gestire liberamente i propri bucket all'interno del rispettivo volume di competenza.
- **Key**: L'oggetto dati effettivo, corrispondente a un file o oggetto S3.

Le celle seguenti illustrano le operazioni di gestione della gerarchia tramite l'interfaccia a riga di comando di Ozone (`ozone sh`).

In [ ]:
# 1. Creazione del Volume amministrativo per il laboratorio
!ozone sh volume create /volume-lab 2>/dev/null || echo "Volume già esistente"

# Ispezione dei metadati del volume creato
!ozone sh volume info /volume-lab 2>/dev/null

In [ ]:
# 2. Creazione di un Bucket dedicato all'interno del Volume
!ozone sh bucket create /volume-lab/dati 2>/dev/null || echo "Bucket già esistente"

# Elenco dei bucket registrati nel volume
!ozone sh bucket list /volume-lab 2>/dev/null

In [ ]:
# 3. Generazione di un file locale e caricamento come Key nel bucket
!echo 'Benvenuti in Apache Ozone 2.2.1!' > file_benvenuto.txt
!ozone sh key put /volume-lab/dati/file_benvenuto.txt file_benvenuto.txt 2>/dev/null

# Ispezione dei metadati della chiave
!ozone sh key info /volume-lab/dati/file_benvenuto.txt 2>/dev/null

In [ ]:
# 4. Lettura del contenuto della chiave direttamente dallo storage Ozone
!ozone sh key cat /volume-lab/dati/file_benvenuto.txt 2>/dev/null
!rm -f file_benvenuto.txt

---
# PARTE 2: Benchmark e Risoluzione dello "Small Files Problem"

### Il Limite Architetturale di HDFS: Lo "Small Files Problem"
Nell'architettura standard di Apache HDFS, il **NameNode** mantiene in memoria volatile (RAM) l'intero namespace del filesystem e la collocazione dei blocchi dati. Ciascun file, directory o blocco comporta un overhead fisso di circa **150 byte di RAM**.
- Il caricamento di 10.000 file da 1 KB occupa circa 10 MB di disco, ma alloca 10.000 record distinti nella memoria del NameNode.
- In scenari reali ad alta frequenza (stream ingestion da Apache Kafka, sensoristica IoT, micro-partizioni analitiche), la moltiplicazione di piccoli file satura la memoria heap del NameNode, innescando prolungate pause di Garbage Collection e compromettendo la disponibilità dell'intero cluster.

### La Soluzione Architetturale di Apache Ozone: RocksDB nell'Ozone Manager
Apache Ozone risolve strutturalmente questo collo di bottiglia:
1. **Separazione delle Competenze**: L'Ozone Manager (OM) gestisce esclusivamente il namespace logico (volumi, bucket, chiavi), delegando allo Storage Container Manager (SCM) la gestione dei container e dei blocchi fisici sottostanti.
2. **Storage Engine Basato su LSM-Tree**: I metadati dell'OM non sono vincolati alla capacità della RAM; vengono persistiti in **RocksDB**, un motore key-value incorporato ad alte prestazioni ottimizzato per memorie di massa non volatili (NVMe/SSD), impiegando la memoria principale unicamente come cache LRU.
3. **Scalabilità Orizzontale a Miliardi di Oggetti**: Questa progettazione consente a un singolo cluster Ozone di gestire agevolmente oltre **10 miliardi di oggetti**, preservando footprint di memoria stabili e prestazioni costanti.

---
### Esecuzione del Test di Carico: Ingestione Parallela di Oggetti di Piccola Dimensione
Di seguito viene valutata la capacità di ingestione del cluster tramite caricamento parallelo concorrente.

#### Esecuzione Sequenziale via CLI (Riferimento Concettuale)
Per completezza di riferimento, l'ingestione tramite CLI nativa in ciclo sequenziale equivale al seguente snippet:
```bash
for i in {1..10000}; do
    echo "dato $i" > file_$i.txt
    ozone sh key put /volume-lab/dati/file_$i.txt file_$i.txt
    rm -f file_$i.txt
done

# Verifica del conteggio oggetti:
ozone sh key list /volume-lab/dati | grep -o '"name"' | wc -l
```
> *Nota operativa: L'invocazione sequenziale del client CLI comporta l'avvio di una nuova JVM per ciascun comando. Per eseguire una misurazione accurata del throughput del sistema di storage, il benchmark sottostante adotta un'ingestione concorrente multi-thread via Python e Boto3.*

In [ ]:
import os
import time
import boto3
from concurrent.futures import ThreadPoolExecutor
from botocore.client import Config

# Parametri di configurazione del benchmark
NUM_FILE_STRESS = 2000  # Scalabile a valori superiori per test estesi
PAYLOAD_1KB = b"X" * 1024  # Payload fisso di 1 Kilobyte per oggetto
STRESS_BUCKET = "stress-test-small-files"

endpoint_url = os.environ.get("OZONE_S3_ENDPOINT", "http://s3g:9878")
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint_url,
    aws_access_key_id="your-access-key",
    aws_secret_access_key="your-secret-key",
    region_name="us-east-1",
    config=Config(s3={"addressing_style": "path"})
)

# Inizializzazione del bucket dedicato al test di carico
try:
    s3.create_bucket(Bucket=STRESS_BUCKET)
    print(f"[INFO] Bucket '{STRESS_BUCKET}' inizializzato con successo.")
except Exception as e:
    print(f"[INFO] Bucket già presente o avviso: {e}")

print(f"[BENCHMARK] Inizio ingestione concorrente di {NUM_FILE_STRESS} oggetti (1 KB cad.) su Ozone...")

def upload_worker(idx):
    key_name = f"small_files/file_{idx:05d}.txt"
    s3.put_object(Bucket=STRESS_BUCKET, Key=key_name, Body=PAYLOAD_1KB)
    return idx

start_time = time.time()

# Esecuzione concorrente tramite pool di thread per simulare client multipli
with ThreadPoolExecutor(max_workers=30) as executor:
    list(executor.map(upload_worker, range(1, NUM_FILE_STRESS + 1)))

elapsed = time.time() - start_time
throughput = NUM_FILE_STRESS / elapsed

print("\n--- Risultati del Benchmark ---")
print(f"  - Oggetti elaborati: {NUM_FILE_STRESS} (payload: 1 KB)")
print(f"  - Tempo totale: {elapsed:.2f} s")
print(f"  - Throughput medio: {throughput:.1f} oggetti/s")
print("  - Metadati OM: gestiti su storage persistente RocksDB con footprint heap costante.")

In [ ]:
# Verifica dell'integrità e del conteggio effettivo degli oggetti nel bucket
paginator = s3.get_paginator('list_objects_v2')
conteggio_totale = 0

for page in paginator.paginate(Bucket=STRESS_BUCKET, Prefix='small_files/'):
    conteggio_totale += len(page.get('Contents', []))

print(f"[VERIFICA] Oggetti rilevati nel bucket '{STRESS_BUCKET}': {conteggio_totale}")

# Ispezione del campione iniziale degli oggetti memorizzati
anteprima = s3.list_objects_v2(Bucket=STRESS_BUCKET, Prefix='small_files/', MaxKeys=5)
print("\nCampione delle prime chiavi memorizzate:")
for obj in anteprima.get('Contents', []):
    print(f"  - Key: {obj['Key']} | Dimensione: {obj['Size']} bytes | ETag: {obj['ETag']}")

In [ ]:
# Pulizia delle risorse allocate per il test di carico
print("[CLEANUP] Rimozione degli oggetti di test in corso...")

def delete_worker(idx):
    key_name = f"small_files/file_{idx:05d}.txt"
    s3.delete_object(Bucket=STRESS_BUCKET, Key=key_name)

with ThreadPoolExecutor(max_workers=30) as executor:
    list(executor.map(delete_worker, range(1, NUM_FILE_STRESS + 1)))

s3.delete_bucket(Bucket=STRESS_BUCKET)
print(f"[INFO] Bucket '{STRESS_BUCKET}' svuotato e deallocato con successo.")

---
# PARTE 3: Compatibilità Nativa con l'API S3

In Apache Ozone, il supporto al protocollo S3 non è implementato come un semplice adapter o layer di emulazione, ma costituisce un'interfaccia di primo livello integrata nel core dell'architettura.

### Mappatura della Gerarchia S3 nel Modello Ozone
Quando le operazioni vengono eseguite tramite le API S3 (con client quali Boto3, AWS CLI, Apache Spark o Trino):
- I bucket S3 risiedono nel volume di sistema dedicato denominato **`s3v`**.
- Un bucket S3 denominato `lab-s3-compatibility` corrisponde univocamente al percorso Ozone `/s3v/lab-s3-compatibility`.
- Un oggetto S3 identificato dalla chiave `dataset/informazioni.json` corrisponde alla chiave Ozone `/s3v/lab-s3-compatibility/dataset/informazioni.json`.

Le celle successive dimostrano la trasparenza di questo mapping creando risorse via API S3 e ispezionandole contestualmente tramite la CLI nativa di Ozone.

In [ ]:
DEMO_BUCKET = "lab-s3-compatibility"

# 1. Creazione del bucket tramite interfaccia S3 (Boto3)
s3.create_bucket(Bucket=DEMO_BUCKET)
print(f"[INFO] Bucket S3 '{DEMO_BUCKET}' creato con successo.")

# 2. Upload dell'oggetto con metadati applicativi personalizzati
s3.put_object(
    Bucket=DEMO_BUCKET,
    Key="dataset/informazioni.json",
    Body=b'{"corso": "Big Data Architecture", "storage": "Apache Ozone", "protocollo": "Amazon S3"}',
    ContentType="application/json",
    Metadata={"autore": "studente", "ambiente": "docker-compose"}
)
print("[INFO] Oggetto caricato via Boto3 con metadati associati.")

In [ ]:
# 3. Dimostrazione di interoperabilità bidirezionale:
# Ispezione del bucket e della chiave creati via S3 mediante la CLI nativa di Ozone (/s3v)

print("[OZONE CLI] Ispezione metadati bucket S3 (percorso: /s3v/lab-s3-compatibility):")
!ozone sh bucket info /s3v/lab-s3-compatibility 2>/dev/null

print("\n[OZONE CLI] Ispezione metadati chiave S3:")
!ozone sh key info /s3v/lab-s3-compatibility/dataset/informazioni.json 2>/dev/null

In [ ]:
# 4. Confronto di lettura: recupero payload via Boto3 e via Ozone CLI
print("[READ S3] Recupero tramite client Boto3 (S3 GET):")
obj_s3 = s3.get_object(Bucket=DEMO_BUCKET, Key="dataset/informazioni.json")
print("  - Content-Type:", obj_s3["ContentType"])
print("  - Metadati custom:", obj_s3["Metadata"])
print("  - Contenuto:", obj_s3["Body"].read().decode("utf-8"))

print("\n[READ OZONE] Recupero tramite CLI nativa ('ozone sh key cat'):")
!ozone sh key cat /s3v/lab-s3-compatibility/dataset/informazioni.json 2>/dev/null

In [ ]:
# 5. Pulizia delle risorse allocate per le dimostrazioni
try:
    paginator = s3.get_paginator('list_objects_v2')
    objects_to_delete = []
    for page in paginator.paginate(Bucket=DEMO_BUCKET):
        for obj in page.get('Contents', []):
            objects_to_delete.append({'Key': obj['Key']})
    if objects_to_delete:
        s3.delete_objects(Bucket=DEMO_BUCKET, Delete={'Objects': objects_to_delete})
        print(f"[CLEANUP] Eliminati {len(objects_to_delete)} oggetti nel bucket S3.")
except Exception as e:
    print(f"[WARN] Notifica durante lo svuotamento: {e}")

s3.delete_bucket(Bucket=DEMO_BUCKET)
print(f"[CLEANUP] Bucket '{DEMO_BUCKET}' deallocato.")

# Pulizia del volume iniziale /volume-lab creato nella Parte 1
!ozone sh bucket delete --recursive /volume-lab/dati 2>/dev/null || true
!ozone sh volume delete /volume-lab 2>/dev/null || true
print("[CLEANUP] Volume '/volume-lab' deallocato.")

---
# PARTE 4: Monitoraggio con la Dashboard Apache Ozone Recon

**Recon** costituisce il servizio integrato di telemetria, diagnostica e analisi di Apache Ozone.

L'interfaccia web di Recon è accessibile localmente al seguente indirizzo:
### **[http://localhost:9888](http://localhost:9888)**

#### Componenti e Metriche Principali di Recon
1. **Overview**: Stato operativo generale del cluster, conteggio dei DataNode attivi, capacità totale, allocata e disponibile.
2. **Datanodes**: Dettagli su ciascun nodo di memorizzazione, latenze di heartbeat e disponibilità disco.
3. **Pipelines**: Stato delle pipeline di replica basate sul protocollo di consenso distribuito **Apache Ratis**.
4. **Containers**: Distribuzione dei container gestiti dallo Storage Container Manager (SCM).
5. **OM DB Insights**: Dimensionamento e distribuzione analitica dei metadati persistiti su RocksDB all'interno dell'Ozone Manager.